### Dataset Collection and Preparation

We curated a dataset of 45 satellite images from multiple sources to improve the model’s generalization capabilities. This data includes satellite images from diverse regions and visual conditions, supporting robustness in model performance. The data sources are as follows:

- **Streets of Astana** ([satellites.pro](https://satellites.pro/Astana_map.Kazakhstan))
- **Open-source Datasets**:
  - [Satellite Images of Water Bodies](https://www.kaggle.com/datasets/franciscoescobar/satellite-images-of-water-bodies)
  - [xView2 Dataset](https://xview2.org/)
  - [Satellite Image Processing on Kaggle](https://www.kaggle.com/code/bloodaxe/deep-learning-for-satellite-image-processing)
- **Google Satellite Images**: Additional images were sourced directly from Google.

#### Data Augmentation
To further enhance the dataset and prevent overfitting, we applied various augmentation techniques, including flipping and rotation. These methods increase the dataset’s diversity, helping the model learn to recognize features from different perspectives and orientations.

#### Data Split
The dataset was divided into training, testing, and validation sets with a 70/20/10 split, providing a balanced approach to training and evaluation.

### Model Performance
Our model achieved an 23.85% Mean Average Precision (mAP@0.5) on the validation set, demonstrating strong performance in segmentation accuracy.

### Visualization of Results on New Images
The following section presents the segmentation results on completely new images of the streets of Astana, showcasing the model’s capability to generalize to unseen data effectively.


# SETUP

In [ ]:
!pip install ultralytics

In [ ]:
# Load the manually labeled dataset by our team
!unzip '/content/Geospatial AI.v1i.yolov11.zip' -d '/content/Geospatial_AI_data'

In [ ]:
# Use yolo11m-seg as a pre-trained model for fine-tuning
!wget https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11m-seg.pt

# TRAIN

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/yolo11m-seg.pt')  # or yolov8s-seg.pt, yolov8m-seg.pt, etc.
model.train(data='/content/Geospatial_AI_data/data.yaml', epochs=50, imgsz=640)

# INFERENCE IMAGE

## Calculating infrastructual data from sample image

### Infrastructural Data Extraction

**Input**:
- `map_size_ratio`: Scaling factor for real-world measurements
- `image`: Satellite image for processing
- `region_id`: Identifier for the region

**Output**:
- Infrastructure data summary for the specified `region_id`


In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np

def segment_and_analyze(map_size_ratio, file_path, region_id, model_path='/content/runs/segment/train/weights/best.pt'):
    # Load the trained model
    model = YOLO(model_path)

    # Perform inference
    results = model(file_path)

    # Save processed image
    processed_path = os.path.splitext(file_path)[0] + '_processed.jpg'
    annotated_image = results[0].plot()
    cv2.imwrite(processed_path, annotated_image)
    print(f"Saved segmented image to {processed_path}")

    # Initialize output dictionary with default fields
    output_data = {
        "region_id": region_id,
        "cracks_count": 0,
        "agriculture": 0,
        "damage": 0,
        "methane": 0,
        "old_road": 0,
        "road": 0,
        "water": 0
    }

    # Iterate over detected instances
    for result in results:
        if result.masks is not None:
            masks = result.masks.data.cpu().numpy()  # Access mask data
            classes = result.boxes.cls.cpu().numpy().astype(int)
            names = result.names

            for cls, mask in zip(classes, masks):
                class_name = names[cls]
                area = np.sum(mask) * map_size_ratio  # Calculate area in real-world units

                if class_name == "crack":
                    output_data["cracks_count"] += 1
                elif class_name in output_data:
                    output_data[class_name] += area

    return output_data

# Example usage
map_size_ratio = 1/10  # Replace with actual ratio
file_path = '/content/sample2.jpg'  # Input image path
region_id = 123  # Example region ID

result = segment_and_analyze(map_size_ratio, file_path, region_id)
print(result)

## Inference results vizualized

In [ ]:
import cv2
import matplotlib.pyplot as plt

def show_original_and_predicted(file_path):
    """
    Display the original image and the predicted (processed) image side by side.

    :param file_path: Path to the original image file
    """
    # Load original image
    original_image = cv2.imread(file_path)
    original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)  # Convert to RGB for display

    # Load processed (segmented) image
    processed_path = os.path.splitext(file_path)[0] + '_processed.jpg'
    segmented_image = cv2.imread(processed_path)
    segmented_image = cv2.cvtColor(segmented_image, cv2.COLOR_BGR2RGB)  # Convert to RGB for display

    # Display images side by side
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(original_image)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title("Segmented Image")
    plt.imshow(segmented_image)
    plt.axis("off")

    plt.show()

# Example usage
file_path = '/content/sample2.jpg'
show_original_and_predicted(file_path)